# GRPO Smoke Run Analysis

This notebook is for understanding whether the 100-step GRPO smoke run was a useful engineering check and whether the reward signal is good enough for the next run.

The main point: **do not judge a GRPO run by the generated text alone and do not judge it by `train_loss` alone.** For this project, the useful signals are:

- Did the job finish and save a checkpoint?
- Did the reward function ever assign positive reward?
- Can the parser extract final numeric answers from model completions?
- Are completions following the requested final-answer format?
- Is the model leaking into a new prompt or generating long unrelated text?
- Is there enough reward variation inside completion groups for GRPO to learn?

For Day 2, the goal is not to prove the model is good at GSM8K. The goal is to decide whether the training pipeline and reward signal are reliable enough to run a slightly larger smoke experiment.

## Current Interpretation Of The 100-Step Run

From the Slurm output already inspected:

- Job `50954114` completed successfully.
- It reached `100/100` optimizer steps.
- It saved `checkpoint-100`.
- Runtime was about 18-19 minutes wall time.
- The printed Step 100 examples were wrong, so their reward `0.00` was correct.

That means the **toolchain is working**: Slurm, CUDA, model load, GSM8K data, GRPOTrainer, checkpoint save.

It does **not** yet mean the training signal is good. The next question is whether the saved completions contain any positive rewards and whether answer extraction/formatting is reliable.

A subtle but important detail: if a printed row has `Advantage = -0.50`, that often means this completion was worse than other completions sampled for the same prompt. So two printed bad completions do not prove the whole group had zero reward. We need the saved completion records to know.

## Metric Guide

Use these metrics as a Day 2 dashboard:

| Metric | What it means | Good for this smoke run | Bad sign |
| --- | --- | --- | --- |
| `final_checkpoint_step` | Last checkpoint saved | `>= 100` | missing checkpoint |
| `completion_rows` | Number of saved completions inspected | `> 0` | no completion artifacts |
| `reward_mean` | Average exact-match reward | any nonzero is useful | exactly zero everywhere |
| `reward_max` | Whether any completion got reward 1 | `1.0` | `0.0` |
| `reward_std` | Whether completions differ in reward | nonzero | zero everywhere |
| `answer_extraction_rate` | Parser found a numeric answer | high, ideally `> 80%` | low means format/parser issue |
| `hash_format_rate` | Completion used `#### answer` style | high is nice | low means prompt not followed |
| `final_phrase_rate` | Completion used phrases like `Final answer:` | useful fallback | zero means weak formatting |
| `prompt_leak_rate` | Completion starts another prompt / unrelated task | low, ideally `< 10%` | high means generation too long or stop tokens bad |
| `completion_chars_mean` | Output length | moderate | huge means rambling/clipping risk |

For this project, **reward quality matters more than loss**. A tiny `train_loss` does not mean the model learned math; it only says the policy-gradient objective had a small average value under this setup.

## Configuration

Edit these paths if you analyze a different run.

In [ ]:
from pathlib import Path
import ast
import json
import re
import sys

import pandas as pd

RUN_DIR = Path('/scratch/huterer_root/huterer0/jiamingp/pqs/ckpts/smoke_qwen2_5_0_5b_grpo_100step')
LOG_DIR = Path('/scratch/huterer_root/huterer0/jiamingp/pqs/repos/posttrain-quant-serve/logs')
JOB_ID = '50954114'

# This works whether Jupyter starts in the repo root or in notebooks/.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'scripts').exists() and (REPO_ROOT.parent / 'scripts').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'scripts'))

from gsm8k_reward import (
    extract_model_answer,
    extract_reference_answer,
    gsm8k_exact_match_reward,
)

print('RUN_DIR:', RUN_DIR, 'exists=', RUN_DIR.exists())
print('LOG_DIR:', LOG_DIR, 'exists=', LOG_DIR.exists())
print('REPO_ROOT:', REPO_ROOT)

## 1. Artifact Checklist

This answers the engineering question: **did the run finish and save something reloadable?**

A good smoke run should have at least one `checkpoint-*` directory and a final model file like `model.safetensors`.

In [ ]:
def file_size_mib(path: Path) -> float:
    return path.stat().st_size / 1024**2

artifact_rows = []
if RUN_DIR.exists():
    for path in sorted(RUN_DIR.iterdir()):
        artifact_rows.append({
            'name': path.name,
            'kind': 'dir' if path.is_dir() else 'file',
            'size_mib': None if path.is_dir() else round(file_size_mib(path), 1),
        })

artifacts = pd.DataFrame(artifact_rows)
display(artifacts)

checkpoint_steps = []
for path in sorted(RUN_DIR.glob('checkpoint-*')) if RUN_DIR.exists() else []:
    match = re.search(r'checkpoint-(\d+)$', path.name)
    if match:
        checkpoint_steps.append(int(match.group(1)))

final_checkpoint_step = max(checkpoint_steps) if checkpoint_steps else None
print('checkpoint_steps:', checkpoint_steps)
print('final_checkpoint_step:', final_checkpoint_step)
print('artifact_status:', 'PASS' if final_checkpoint_step and final_checkpoint_step >= 100 else 'WARN/FAIL')

## 2. Slurm Log Summary

This answers: **did the trainer really reach 100 steps, and were there warnings/errors?**

Look for:

- progress line like `100/100`
- training summary dictionary with `train_runtime`
- warnings that did not stop the job
- actual Python tracebacks or nonzero exit codes

In [ ]:
out_files = sorted(LOG_DIR.glob(f'*-{JOB_ID}.out')) if JOB_ID else sorted(LOG_DIR.glob('pqs-grpo-100-*.out'))[-3:]
err_files = sorted(LOG_DIR.glob(f'*-{JOB_ID}.err')) if JOB_ID else sorted(LOG_DIR.glob('pqs-grpo-100-*.err'))[-3:]

print('OUT files:')
for path in out_files:
    print(' ', path)
print('ERR files:')
for path in err_files:
    print(' ', path)

summary_records = []
for path in out_files:
    text = path.read_text(errors='replace')
    for line in text.splitlines():
        if line.startswith("{'train_runtime'"):
            try:
                summary_records.append({'file': path.name, **ast.literal_eval(line)})
            except Exception as exc:
                summary_records.append({'file': path.name, 'parse_error': str(exc), 'raw': line})

summary = pd.DataFrame(summary_records)
display(summary)

for path in err_files:
    text = path.read_text(errors='replace')
    print('\n=== warnings/errors from', path.name, '===')
    for line in text.splitlines():
        lower = line.lower()
        if any(key in lower for key in ['warning', 'traceback', 'error', '100/100', 'writing model shards']):
            print(line[:240])

## 3. Load Completion Artifacts

This is the most important section.

The Slurm table only prints a few examples. TRL also writes completion records under `RUN_DIR/completions/`. The exact file format can vary, so this loader tries JSONL, JSON, CSV, TSV, and Parquet.

If this section finds zero rows, the run still completed, but we cannot judge reward quality from the notebook yet. In that case, inspect the raw files listed below or adjust the loader for the observed format.

In [ ]:
completion_dir = RUN_DIR / 'completions'
completion_files = sorted([p for p in completion_dir.rglob('*') if p.is_file()]) if completion_dir.exists() else []

file_rows = []
for path in completion_files:
    file_rows.append({
        'file': str(path),
        'suffix': path.suffix,
        'size_kib': round(path.stat().st_size / 1024, 1),
    })

display(pd.DataFrame(file_rows))

def read_completion_file(path: Path) -> pd.DataFrame | None:
    suffix = path.suffix.lower()
    try:
        if suffix == '.jsonl':
            rows = []
            for line in path.read_text(errors='replace').splitlines():
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
            return pd.DataFrame(rows)
        if suffix == '.json':
            obj = json.loads(path.read_text(errors='replace'))
            if isinstance(obj, list):
                return pd.DataFrame(obj)
            if isinstance(obj, dict):
                # Common shapes: list under a top-level key, dict-of-lists, or one record.
                for value in obj.values():
                    if isinstance(value, list) and value and isinstance(value[0], dict):
                        return pd.DataFrame(value)
                try:
                    return pd.DataFrame(obj)
                except ValueError:
                    return pd.DataFrame([obj])
        if suffix == '.csv':
            return pd.read_csv(path)
        if suffix == '.tsv':
            return pd.read_csv(path, sep='\t')
        if suffix == '.parquet':
            return pd.read_parquet(path)
    except Exception as exc:
        print('Could not read', path, ':', repr(exc))
    return None

frames = []
for path in completion_files:
    df = read_completion_file(path)
    if df is not None and len(df) > 0:
        df = df.copy()
        df['source_file'] = str(path)
        step_match = re.search(r'(?:step|checkpoint|global_step|_)(\d+)', path.stem)
        if step_match and 'step' not in df.columns:
            df['step'] = int(step_match.group(1))
        frames.append(df)

completions = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print('completion_rows:', len(completions))
print('columns:', list(completions.columns))
display(completions.head(3))

## 4. Compute Reward And Formatting Metrics

This section turns raw completions into useful diagnostics.

Important distinction:

- `answer_extraction_rate` asks whether the parser can find a final number.
- `reward_mean` asks whether that final number matches the GSM8K target.

If extraction is low, the model/prompt/parser interface is broken. If extraction is high but reward is low, the model is simply answering incorrectly.

In [ ]:
def pick_column(df: pd.DataFrame, exact=(), contains=()) -> str | None:
    if df.empty:
        return None
    lower_to_col = {str(c).lower(): c for c in df.columns}
    for name in exact:
        if name.lower() in lower_to_col:
            return lower_to_col[name.lower()]
    for col in df.columns:
        low = str(col).lower()
        if any(key in low for key in contains):
            return col
    return None

completion_col = pick_column(
    completions,
    exact=('completion', 'completions', 'response', 'output', 'text'),
    contains=('completion', 'response', 'output'),
)
prompt_col = pick_column(completions, exact=('prompt',), contains=('prompt',))
answer_col = pick_column(completions, exact=('answer', 'reference_answer', 'target'), contains=('reference', 'target', 'answer'))
reward_cols = [c for c in completions.columns if 'reward' in str(c).lower()]
advantage_col = pick_column(completions, exact=('advantage',), contains=('advantage',))

print('completion_col:', completion_col)
print('prompt_col:', prompt_col)
print('answer_col:', answer_col)
print('reward_cols:', reward_cols)
print('advantage_col:', advantage_col)

analysis = completions.copy()
if len(analysis) and completion_col:
    analysis['_completion_text'] = analysis[completion_col].astype(str)
    analysis['parsed_answer'] = analysis['_completion_text'].map(extract_model_answer)
    analysis['has_extracted_answer'] = analysis['parsed_answer'].notna()
    analysis['uses_hash_answer_format'] = analysis['_completion_text'].str.contains(r'####\s*[-+]?\d|####\s*<answer>', case=False, regex=True, na=False)
    analysis['uses_final_phrase'] = analysis['_completion_text'].str.contains(r'final\s+(numeric\s+)?answer|the\s+answer\s+is|answer\s+is', case=False, regex=True, na=False)
    analysis['completion_chars'] = analysis['_completion_text'].str.len()
    analysis['prompt_leak'] = analysis['_completion_text'].str.contains(r'Human:|Assistant:|Problem:|Solve the math problem|</div>|Question:', case=False, regex=True, na=False)

    if answer_col:
        analysis['target_answer'] = analysis[answer_col].astype(str).map(extract_reference_answer)
        analysis['parsed_exact_match_reward'] = (
            analysis['parsed_answer'].notna()
            & analysis['target_answer'].notna()
            & (analysis['parsed_answer'] == analysis['target_answer'])
        ).astype(float)

    for col in reward_cols:
        analysis[col] = pd.to_numeric(analysis[col], errors='ignore')

metric = {}
metric['completion_rows'] = len(analysis)
metric['completion_col_found'] = completion_col is not None
metric['answer_col_found'] = answer_col is not None

if len(analysis) and completion_col:
    metric['answer_extraction_rate'] = analysis['has_extracted_answer'].mean()
    metric['hash_format_rate'] = analysis['uses_hash_answer_format'].mean()
    metric['final_phrase_rate'] = analysis['uses_final_phrase'].mean()
    metric['prompt_leak_rate'] = analysis['prompt_leak'].mean()
    metric['completion_chars_mean'] = analysis['completion_chars'].mean()
    metric['completion_chars_p95'] = analysis['completion_chars'].quantile(0.95)

if 'parsed_exact_match_reward' in analysis.columns:
    metric['parsed_reward_mean'] = analysis['parsed_exact_match_reward'].mean()
    metric['parsed_reward_max'] = analysis['parsed_exact_match_reward'].max()
    metric['parsed_reward_std'] = analysis['parsed_exact_match_reward'].std()

for col in reward_cols:
    numeric = pd.to_numeric(analysis[col], errors='coerce')
    if numeric.notna().any():
        metric[f'{col}_mean'] = numeric.mean()
        metric[f'{col}_max'] = numeric.max()
        metric[f'{col}_std'] = numeric.std()

metrics = pd.DataFrame([{'metric': k, 'value': v} for k, v in metric.items()])
display(metrics)

## 5. Automatic Smoke-Run Verdict

This verdict is intentionally conservative.

- **Pipeline PASS** means Slurm/model/data/checkpoint path worked.
- **Learning-signal PASS** means there is at least some positive reward or reward variation to train from.
- **Formatting WARN/FAIL** means the model is not reliably producing parseable final answers.

For a 0.5B model on only 10 GSM8K examples, low accuracy is expected. The red flag is not low accuracy; the red flag is **no usable reward signal**.

In [ ]:
def get_metric(name, default=None):
    return metric.get(name, default)

verdict_rows = []

def add_verdict(area, status, evidence, action):
    verdict_rows.append({
        'area': area,
        'status': status,
        'evidence': evidence,
        'recommended_action': action,
    })

if final_checkpoint_step and final_checkpoint_step >= 100:
    add_verdict('pipeline', 'PASS', f'checkpoint-{final_checkpoint_step} exists', 'Keep this environment; do not rebuild it.')
else:
    add_verdict('pipeline', 'FAIL', f'final_checkpoint_step={final_checkpoint_step}', 'Fix checkpoint/run completion before more training.')

if get_metric('completion_rows', 0) == 0:
    add_verdict('completion logging', 'WARN', 'No parsed completion rows found', 'Inspect raw files under completions/ or adjust loader.')
else:
    add_verdict('completion logging', 'PASS', f"{get_metric('completion_rows')} rows loaded", 'Use these rows for reward/prompt diagnostics.')

reward_mean = get_metric('parsed_reward_mean')
reward_max = get_metric('parsed_reward_max')
reward_std = get_metric('parsed_reward_std')
if reward_mean is None:
    reward_metric_names = [k for k in metric if k.endswith('_mean') and 'reward' in k]
    if reward_metric_names:
        base = reward_metric_names[0][:-5]
        reward_mean = get_metric(base + '_mean')
        reward_max = get_metric(base + '_max')
        reward_std = get_metric(base + '_std')

if reward_mean is None:
    add_verdict('reward signal', 'UNKNOWN', 'No target answer or reward column found', 'Load a completion file with answers/rewards, or join completions to GSM8K targets.')
elif reward_max and reward_max > 0:
    add_verdict('reward signal', 'PASS', f'mean={reward_mean:.4f}, max={reward_max:.4f}, std={reward_std:.4f}', 'Run a modest next smoke: 300 steps / 50 examples.')
elif reward_std and reward_std > 0:
    add_verdict('reward signal', 'WARN', f'mean={reward_mean:.4f}, max={reward_max:.4f}, std={reward_std:.4f}', 'Reward varies but no positive exact matches; inspect parser and examples.')
else:
    add_verdict('reward signal', 'FAIL', f'mean={reward_mean:.4f}, max={reward_max:.4f}, std={reward_std:.4f}', 'Do not scale. Fix prompt/reward/parser and rerun 5 steps.')

extract_rate = get_metric('answer_extraction_rate')
if extract_rate is not None:
    if extract_rate >= 0.8:
        add_verdict('answer extraction', 'PASS', f'{extract_rate:.1%}', 'Parser is probably usable.')
    elif extract_rate >= 0.4:
        add_verdict('answer extraction', 'WARN', f'{extract_rate:.1%}', 'Improve prompt format or parser before scaling.')
    else:
        add_verdict('answer extraction', 'FAIL', f'{extract_rate:.1%}', 'The model/parser interface is broken; fix this first.')

leak_rate = get_metric('prompt_leak_rate')
if leak_rate is not None:
    if leak_rate <= 0.1:
        add_verdict('prompt leakage', 'PASS', f'{leak_rate:.1%}', 'Generation length/stop behavior looks acceptable.')
    elif leak_rate <= 0.3:
        add_verdict('prompt leakage', 'WARN', f'{leak_rate:.1%}', 'Consider shorter max completion length or stop handling.')
    else:
        add_verdict('prompt leakage', 'FAIL', f'{leak_rate:.1%}', 'Outputs are drifting into unrelated text; fix generation settings.')

verdict = pd.DataFrame(verdict_rows)
display(verdict)

## 6. Inspect Examples By Reward

Use this to understand *why* the metrics say good or bad.

Look at:

- `parsed_answer`: what the reward code thinks the model answered
- `target_answer`: GSM8K target, if available
- reward/advantage columns
- whether the completion leaks into another prompt or unrelated task

If a completion looks correct but reward is zero, add that format to `scripts/check_reward_parser.py` and patch `scripts/gsm8k_reward.py`.

In [ ]:
if len(analysis) == 0 or completion_col is None:
    print('No completion table available to inspect.')
else:
    show_cols = []
    for col in ['step', prompt_col, completion_col, 'parsed_answer', 'target_answer', 'parsed_exact_match_reward', advantage_col, 'prompt_leak', 'source_file']:
        if col and col in analysis.columns and col not in show_cols:
            show_cols.append(col)
    for col in reward_cols:
        if col not in show_cols:
            show_cols.append(col)

    sort_col = 'parsed_exact_match_reward' if 'parsed_exact_match_reward' in analysis.columns else (reward_cols[0] if reward_cols else None)
    if sort_col:
        display(analysis.sort_values(sort_col, ascending=False)[show_cols].head(10))
        display(analysis.sort_values(sort_col, ascending=True)[show_cols].head(10))
    else:
        display(analysis[show_cols].head(10))

## 7. Parser Sandbox

Paste a suspicious model completion here. This tells you what the reward code extracts.

Example interpretation:

- If the parser extracts the wrong number, the reward parser is too naive.
- If the parser extracts `None`, the model is not giving a parseable answer.
- If the parser extracts the right number but reward is zero, the reference target may not be loaded correctly.

In [ ]:
sample_completion = "Final answer: 312 pages per year."
sample_reference = '... #### 624'

print('parsed model answer:', extract_model_answer(sample_completion))
print('parsed reference answer:', extract_reference_answer(sample_reference))
print('reward:', gsm8k_exact_match_reward([sample_completion], [sample_reference]))

## 8. What To Do Next

Use the verdict above:

### If pipeline PASS and reward signal PASS

Run the next small smoke, still on Qwen2.5-0.5B-Instruct:

```bash
sbatch \
  --job-name=pqs-grpo-300 \
  --account=cavestru0 \
  --partition=spgpu \
  --gres=gpu:1 \
  --cpus-per-task=4 \
  --mem=32G \
  --time=01:30:00 \
  --output=logs/%x-%j.out \
  --error=logs/%x-%j.err \
  --export=ALL,MAX_STEPS=300,DATASET_LIMIT=50,OUTPUT_DIR=/scratch/huterer_root/huterer0/jiamingp/pqs/ckpts/qwen2_5_0_5b_grpo_300step_50gsm8k \
  slurm/smoke_single_gpu.sbatch
```

### If reward signal FAIL

Do **not** scale. Fix one of these first:

- prompt asks for a cleaner final line
- parser recognizes the model's actual answer style
- max completion length is shorter so the model does not drift into another prompt
- reward function adds a small format reward in addition to exact-match reward

### If answer extraction FAIL but some outputs contain obvious answers

Patch `scripts/gsm8k_reward.py`, add cases to `scripts/check_reward_parser.py`, run the parser test, then rerun a 5-step job.